# Marconi100 Telemetry — Pré-processamento e Engenharia de Atributos
**Classificação de Risco de Desperdício Ambiental em Datacenters Reais**

Este notebook apresenta a análise exploratória, engenharia de atributos térmicos e elétricos, e validação do modelo baseline sobre dados reais do supercomputador **Marconi100 (CINECA)**, extraídos do dataset oficial **M100 ExaData** (Zenodo Record [7588815](https://doi.org/10.5281/zenodo.7588815)).

Trabalho desenvolvido como continuidade e validação empírica do artigo de *Coelho et al. (SACI 2026)*.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Bibliotecas importadas com sucesso!')

## 1. Carregamento da Base Processada
Lemos a base tratada gerada pelo pipeline (`base_processada.parquet`), que já unifica as medições de potência, diferenciais térmicos e a rotulação das classes de risco.

In [ ]:
caminho_dados = Path('../resultados/base_processada.parquet')
if not caminho_dados.exists():
    caminho_dados = Path('resultados/base_processada.parquet')

df = pd.read_parquet(caminho_dados)
print(f'Total de instâncias: {len(df):,}')
print(f'Total de colunas: {df.shape[1]}')
df[['timestamp', 'potencia_total_w', 'temperatura_gpu_media_c', 'delta_t_gpu_inlet', 'classe_risco_desperdicio']].head()

## 2. Distribuição da Classe de Risco de Desperdício Ambiental

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x='classe_risco_desperdicio', order=['Baixo', 'Moderado', 'Alto'], palette='viridis')
plt.title('Distribuição da Classe de Risco de Desperdício (Marconi100)')
plt.xlabel('Nível de Risco')
plt.ylabel('Instâncias')
plt.tight_layout()
plt.show()

print(df['classe_risco_desperdicio'].value_counts(normalize=True).round(3))

## 3. Dispersão: Potência Ativa (W) vs Temperatura Média de GPU (°C)
Evidencia os regimes operacionais em que consumo elétrico elevado ocorre com ou sem estresse de temperatura nos nós.

In [ ]:
sample = df.sample(min(3000, len(df)), random_state=42)
plt.figure(figsize=(9, 5))
sns.scatterplot(
    data=sample,
    x='potencia_total_w',
    y='temperatura_gpu_media_c',
    hue='classe_risco_desperdicio',
    palette='viridis',
    alpha=0.6
)
plt.title('Relação Potência Total (W) x Temperatura Média de GPU (°C)')
plt.xlabel('Potência Total do Nó (Watts)')
plt.ylabel('Temperatura Média de GPU (°C)')
plt.tight_layout()
plt.show()

## 4. Avaliação do Diferencial Térmico (ΔT = T_gpu - T_inlet)
Mede a eficácia de arrefecimento: quanto maior o diferencial térmico por Watt consumido, maior a ineficiência de refrigeração.

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x='classe_risco_desperdicio', y='delta_t_gpu_inlet', order=['Baixo', 'Moderado', 'Alto'], palette='Set2')
plt.title('Diferencial Térmico GPU - Entrada (ΔT) por Nível de Risco')
plt.xlabel('Nível de Risco')
plt.ylabel('ΔT (°C)')
plt.tight_layout()
plt.show()

## 5. Inspeção do Modelo Baseline Random Forest
Carregamos o classificador treinado pelo pipeline para validação imediata pelo Gregory.

In [ ]:
caminho_modelo = Path('../resultados/modelo_baseline.joblib')
if not caminho_modelo.exists():
    caminho_modelo = Path('resultados/modelo_baseline.joblib')

modelo = joblib.load(caminho_modelo)
print('Classificador Baseline Carregado com Sucesso!')
print(modelo)